In [ ]:
#from transformers import AutoModel, AutoTokenizer

#tokenizer = AutoTokenizer.from_pretrained("BioMistral/BioMistral-7B")
#model = AutoModel.from_pretrained("BioMistral/BioMistral-7B")

In [ ]:
# # Instalar librerías necesarias
# !pip install biopython requests transformers beautifulsoup4 --quiet

# from Bio import Entrez
# import requests
# from transformers import T5ForConditionalGeneration, T5Tokenizer
# import json
# import time
# from bs4 import BeautifulSoup
# import torch

# # Configuración
# Entrez.email = "apaolahernandezgonzalez@gmail.com"  # Reemplaza con tu correo
# DEEPL_API_KEY = "36666922-7e45-4058-bc5c-f1c07d8429d0:fx"  # Reemplaza con tu clave API de DeepL

# # Función para extraer el resumen del XML de PMC
# def extract_abstract(xml_text):
#     try:
#         soup = BeautifulSoup(xml_text, "xml")
#         abstract_tag = soup.find("abstract")
#         if abstract_tag:
#             abstract_text = " ".join(abstract_tag.get_text().split())
#             abstract_text = abstract_text.replace("Highlights", "").replace("Abstract", "")
#             return abstract_text.strip()
#         else:
#             return None
#     except Exception as e:
#         print(f"Error al parsear XML: {e}")
#         return None

# # Función para descargar y extraer resúmenes de PMC
# def fetch_pmc_abstracts(query, max_results=2):
#     try:
#         handle = Entrez.esearch(db="pmc", term=query, retmax=max_results)
#         record = Entrez.read(handle)
#         handle.close()

#         if not record.get("IdList"):
#             print("No se encontraron artículos en PMC.")
#             return []

#         abstracts_en = []
#         for id in record["IdList"]:
#             try:
#                 handle = Entrez.efetch(db="pmc", id=id, rettype="abstract", retmode="xml")
#                 xml_text = handle.read().strip()
#                 handle.close()

#                 abstract_text = extract_abstract(xml_text)
#                 if abstract_text and len(abstract_text.split()) > 20:
#                     abstracts_en.append(abstract_text)
#                 time.sleep(0.3)  # Evitar bloqueo
#             except Exception as e:
#                 print(f"Error al descargar ID {id}: {e}")
#         return abstracts_en
#     except Exception as e:
#         print(f"Error en búsqueda PMC: {e}")
#         return []

# # Función para traducir con DeepL API (usando headers)
# def translate_to_spanish(text):
#     try:
#         url = "https://api-free.deepl.com/v2/translate"
#         headers = {
#             "Authorization": f"DeepL-Auth-Key {DEEPL_API_KEY}",
#             "Content-Type": "application/json"
#         }
#         data = {
#             "text": [text],
#             "target_lang": "ES"
#         }
#         response = requests.post(url, headers=headers, json=data)
#         if response.status_code == 200:
#             return response.json()["translations"][0]["text"]
#         else:
#             print(f"Error en DeepL API: {response.status_code} - {response.text}")
#             return text
#     except Exception as e:
#         print(f"Error en traducción: {e}")
#         return text

# # Función para generar preguntas en español
# def generate_questions(text):
#     try:
#         model_name = "mrm8488/t5-base-finetuned-question-generation-ap"
#         tokenizer = T5Tokenizer.from_pretrained(model_name)
#         model = T5ForConditionalGeneration.from_pretrained(model_name)

#         sentences = [s.strip() for s in text.split('.') if len(s.strip().split()) > 5]
#         questions = []

#         for sentence in sentences[:3]:
#             # Usar un prompt en español para forzar la generación en español
#             input_text = f"generar pregunta en español: {sentence}"
#             inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)
#             outputs = model.generate(**inputs, max_length=64)
#             question = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

#             # Filtrar preguntas válidas
#             if question.endswith('?') and len(question.split()) > 3 and "generar pregunta" not in question:
#                 questions.append(question)
#         return questions
#     except Exception as e:
#         print(f"Error al generar preguntas: {e}")
#         return []

# # Función principal
# def create_medical_dataset(query, max_articles=2):
#     print(f"Buscando artículos sobre: {query}")
#     abstracts_en = fetch_pmc_abstracts(query, max_articles)

#     dataset = []
#     for i, abstract_en in enumerate(abstracts_en):
#         print(f"\n--- Artículo {i+1} (inglés) ---")
#         print(abstract_en[:200] + "...")

#         # Traducir al español
#         abstract_es = translate_to_spanish(abstract_en)
#         print(f"\n--- Artículo {i+1} (español) ---")
#         print(abstract_es[:200] + "...")

#         # Generar preguntas en español
#         questions = generate_questions(abstract_es)
#         print("Preguntas generadas:", questions)

#         for q in questions:
#             dataset.append({
#                 "prompt": q,
#                 "completion": abstract_es
#             })

#     return dataset

# # Ejemplo de uso
# query = "dengue symptoms"
# dataset = create_medical_dataset(query, max_articles=2)

# # Guardar resultados
# filename = "dataset_medico_es.json"
# with open(filename, "w", encoding="utf-8") as f:
#     json.dump(dataset, f, ensure_ascii=False, indent=2)

# print(f"\nDataset guardado como '{filename}' (total: {len(dataset)} ejemplos).")
# print("Descarga el archivo desde el panel izquierdo de Colab (sección 'Files').")

In [ ]:
import json
import random

# Lista de síntomas comunes en español
sintomas = [
    "dolor de cabeza", "fiebre", "mareos", "náuseas", "dolor de garganta",
    "tos", "dolor abdominal", "dolor muscular", "fatiga", "dolor de pecho",
    "dificultad para respirar", "visión borrosa", "diarrea", "vómitos", "pérdida del olfato"
]

# Lista de preguntas típicas de un médico
preguntas_medico = [
    "¿Qué síntomas tienes?",
    "¿Desde cuándo tienes {sintoma}?",
    "¿El {sintoma} es constante o viene y va?",
    "¿Has tenido fiebre?",
    "¿Tienes algún otro síntoma?",
    "¿Has tomado algún medicamento?",
    "¿Tienes alergias a algún medicamento?",
    "¿Has estado en contacto con alguien enfermo?",
    "¿Tienes alguna condición médica preexistente?",
    "¿Has viajado recientemente?"
]

# Lista de respuestas típicas de un paciente
respuestas_paciente = [
    "Sí, desde hace {dias} días.",
    "No, no he tenido {sintoma}.",
    "Sí, es constante.",
    "No, viene y va.",
    "Sí, tomé {medicamento}.",
    "No, no he tomado nada.",
    "Sí, tengo alergia a {alergia}.",
    "No, no tengo alergias.",
    "Sí, estuve con alguien enfermo.",
    "No, no he estado en contacto con nadie enfermo.",
    "Sí, tengo {condicion}.",
    "No, no tengo condiciones médicas.",
    "Sí, viajé a {lugar}.",
    "No, no he viajado."
]

# Lista de acciones o diagnósticos típicos
acciones_diagnostico = [
    "Vamos a tomar tu presión.",
    "Necesito revisar tu temperatura.",
    "Voy a auscultar tus pulmones.",
    "Vamos a hacer un análisis de sangre.",
    "Necesito que te hagas una radiografía.",
    "Voy a recetarte {medicamento}.",
    "Debes descansar y tomar líquidos.",
    "Vuelve en {dias} días si no mejoras.",
    "Necesitas ir al hospital.",
    "Voy a derivarte con un especialista."
]

# Lista de medicamentos comunes
medicamentos = [
    "paracetamol", "ibuprofeno", "amoxicilina", "loratadina",
    "omeprazol", "salbutamol", "azitromicina", "acetaminofén"
]

# Lista de condiciones médicas
condiciones = [
    "diabetes", "hipertensión", "asma", "alergias",
    "problemas cardíacos", "migrañas", "artritis", "gastritis"
]

# Lista de lugares
lugares = [
    "la playa", "la montaña", "otro país", "la ciudad",
    "el campo", "el extranjero"
]

# Función para generar un diálogo médico-paciente
def generar_dialogo():
    dialogo = []
    sintoma_actual = random.choice(sintomas)
    dias = random.randint(1, 7)
    medicamento = random.choice(medicamentos)
    condicion = random.choice(condiciones)
    lugar = random.choice(lugares)

    # Inicio del diálogo
    dialogo.append({
        "speaker": "médico",
        "text": random.choice([
            "Hola, ¿en qué puedo ayudarte hoy?",
            "Buenos días, ¿qué te trae por aquí?",
            "Cuéntame, ¿qué síntomas tienes?"
        ])
    })

    # Paciente describe síntoma inicial
    dialogo.append({
        "speaker": "paciente",
        "text": random.choice([
            f"Doctor, tengo {sintoma_actual}.",
            f"Me duele mucho {sintoma_actual.replace('dolor de ', '')}.",
            f"Desde hace {dias} días tengo {sintoma_actual}."
        ])
    })

    # Médico hace preguntas
    for _ in range(random.randint(2, 4)):
        pregunta = random.choice(preguntas_medico)
        if "{sintoma}" in pregunta:
            pregunta = pregunta.replace("{sintoma}", sintoma_actual)
        dialogo.append({"speaker": "médico", "text": pregunta})

        # Paciente responde
        respuesta = random.choice(respuestas_paciente)
        if "{sintoma}" in respuesta:
            respuesta = respuesta.replace("{sintoma}", sintoma_actual)
        if "{dias}" in respuesta:
            respuesta = respuesta.replace("{dias}", str(dias))
        if "{medicamento}" in respuesta:
            respuesta = respuesta.replace("{medicamento}", medicamento)
        if "{alergia}" in respuesta:
            respuesta = respuesta.replace("{alergia}", condicion)
        if "{condicion}" in respuesta:
            respuesta = respuesta.replace("{condicion}", condicion)
        if "{lugar}" in respuesta:
            respuesta = respuesta.replace("{lugar}", lugar)
        dialogo.append({"speaker": "paciente", "text": respuesta})

    # Médico realiza acción o diagnóstico
    accion = random.choice(acciones_diagnostico)
    if "{medicamento}" in accion:
        accion = accion.replace("{medicamento}", medicamento)
    if "{dias}" in accion:
        accion = accion.replace("{dias}", str(random.randint(3, 10)))
    dialogo.append({"speaker": "médico", "text": accion})

    # Cierre del diálogo
    dialogo.append({
        "speaker": "médico",
        "text": random.choice([
            "Si los síntomas persisten, vuelve a consultar.",
            "Cuídate y toma tus medicamentos.",
            "Espero que te mejores pronto.",
            "Mantente en reposo y hidratado."
        ])
    })

    return dialogo

# Función para convertir diálogos en formato prompt/completion
def dialogos_a_dataset(num_dialogos=10):
    dataset = []
    for _ in range(num_dialogos):
        dialogo = generar_dialogo()
        # Convertir diálogo en pares de pregunta/respuesta
        for i in range(len(dialogo) - 1):
            prompt = dialogo[i]["text"]
            completion = dialogo[i+1]["text"]
            if dialogo[i]["speaker"] == "paciente" and dialogo[i+1]["speaker"] == "médico":
                dataset.append({
                    "prompt": prompt,
                    "completion": completion
                })
    return dataset

# Generar dataset
dataset = dialogos_a_dataset(num_dialogos=20)

# Guardar resultados
filename = "dataset_dialogos_medicos_es.json"
with open(filename, "w", encoding="utf-8") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)

print(f"\nDataset guardado como '{filename}' (total: {len(dataset)} ejemplos).")
print("Descarga el archivo desde el panel izquierdo de Colab (sección 'Files').")


Dataset guardado como 'dataset_dialogos_medicos_es.json' (total: 84 ejemplos).
Descarga el archivo desde el panel izquierdo de Colab (sección 'Files').


In [ ]:
import json
import random

# Lista de síntomas comunes en español
sintomas = [
    "dolor de cabeza", "fiebre", "mareos", "náuseas", "dolor de garganta",
    "tos", "dolor abdominal", "dolor muscular", "fatiga", "dolor de pecho",
    "dificultad para respirar", "visión borrosa", "diarrea", "vómitos", "pérdida del olfato"
]

# Lista de preguntas típicas de un médico
preguntas_medico = [
    "¿Qué síntomas tienes?",
    "¿Desde cuándo tienes {sintoma}?",
    "¿El {sintoma} es constante o viene y va?",
    "¿Has tenido fiebre?",
    "¿Tienes algún otro síntoma?",
    "¿Has tomado algún medicamento?",
    "¿Tienes alergias a algún medicamento?",
    "¿Has estado en contacto con alguien enfermo?",
    "¿Tienes alguna condición médica preexistente?",
    "¿Has viajado recientemente?"
]

# Lista de respuestas típicas de un paciente
respuestas_paciente = [
    "Sí, desde hace {dias} días.",
    "No, no he tenido {sintoma}.",
    "Sí, es constante.",
    "No, viene y va.",
    "Sí, tomé {medicamento}.",
    "No, no he tomado nada.",
    "Sí, tengo alergia a {alergia}.",
    "No, no tengo alergias.",
    "Sí, estuve con alguien enfermo.",
    "No, no he estado en contacto con nadie enfermo.",
    "Sí, tengo {condicion}.",
    "No, no tengo condiciones médicas.",
    "Sí, viajé a {lugar}.",
    "No, no he viajado."
]

# Lista de acciones o diagnósticos típicos
acciones_diagnostico = [
    "Vamos a tomar tu presión.",
    "Necesito revisar tu temperatura.",
    "Voy a auscultar tus pulmones.",
    "Vamos a hacer un análisis de sangre.",
    "Necesito que te hagas una radiografía.",
    "Voy a recetarte {medicamento}.",
    "Debes descansar y tomar líquidos.",
    "Vuelve en {dias} días si no mejoras.",
    "Necesitas ir al hospital.",
    "Voy a derivarte con un especialista."
]

# Lista de medicamentos comunes
medicamentos = [
    "paracetamol", "ibuprofeno", "amoxicilina", "loratadina",
    "omeprazol", "salbutamol", "azitromicina", "acetaminofén"
]

# Lista de condiciones médicas
condiciones = [
    "diabetes", "hipertensión", "asma", "alergias",
    "problemas cardíacos", "migrañas", "artritis", "gastritis"
]

# Lista de lugares
lugares = [
    "la playa", "la montaña", "otro país", "la ciudad",
    "el campo", "el extranjero"
]

# Función para generar un diálogo médico-paciente
def generar_dialogo():
    dialogo = []
    sintoma_actual = random.choice(sintomas)
    dias = random.randint(1, 7)
    medicamento = random.choice(medicamentos)
    condicion = random.choice(condiciones)
    lugar = random.choice(lugares)

    # Inicio del diálogo
    dialogo.append({
        "speaker": "médico",
        "text": random.choice([
            "Hola, ¿en qué puedo ayudarte hoy?",
            "Buenos días, ¿qué te trae por aquí?",
            "Cuéntame, ¿qué síntomas tienes?"
        ])
    })

    # Paciente describe síntoma inicial
    dialogo.append({
        "speaker": "paciente",
        "text": random.choice([
            f"Doctor, tengo {sintoma_actual}.",
            f"Me duele mucho {sintoma_actual.replace('dolor de ', '')}.",
            f"Desde hace {dias} días tengo {sintoma_actual}."
        ])
    })

    # Médico hace preguntas
    for _ in range(random.randint(2, 4)):
        pregunta = random.choice(preguntas_medico)
        if "{sintoma}" in pregunta:
            pregunta = pregunta.replace("{sintoma}", sintoma_actual)
        dialogo.append({"speaker": "médico", "text": pregunta})

        # Paciente responde
        respuesta = random.choice(respuestas_paciente)
        if "{sintoma}" in respuesta:
            respuesta = respuesta.replace("{sintoma}", sintoma_actual)
        if "{dias}" in respuesta:
            respuesta = respuesta.replace("{dias}", str(dias))
        if "{medicamento}" in respuesta:
            respuesta = respuesta.replace("{medicamento}", medicamento)
        if "{alergia}" in respuesta:
            respuesta = respuesta.replace("{alergia}", condicion)
        if "{condicion}" in respuesta:
            respuesta = respuesta.replace("{condicion}", condicion)
        if "{lugar}" in respuesta:
            respuesta = respuesta.replace("{lugar}", lugar)
        dialogo.append({"speaker": "paciente", "text": respuesta})

    # Médico realiza acción o diagnóstico
    accion = random.choice(acciones_diagnostico)
    if "{medicamento}" in accion:
        accion = accion.replace("{medicamento}", medicamento)
    if "{dias}" in accion:
        accion = accion.replace("{dias}", str(random.randint(3, 10)))
    dialogo.append({"speaker": "médico", "text": accion})

    # Cierre del diálogo
    dialogo.append({
        "speaker": "médico",
        "text": random.choice([
            "Si los síntomas persisten, vuelve a consultar.",
            "Cuídate y toma tus medicamentos.",
            "Espero que te mejores pronto.",
            "Mantente en reposo y hidratado."
        ])
    })

    return dialogo

# Función para convertir diálogos en formato de lista de mensajes
def dialogos_a_dataset(num_dialogos=10):
    dataset1 = []
    for _ in range(num_dialogos):
        dialogo = generar_dialogo()
        # Convertir el diálogo en una lista de mensajes con roles adecuados
        conversation_messages = []
        for i in range(len(dialogo)):
            speaker = "user" if dialogo[i]["speaker"] == "paciente" else "assistant"
            conversation_messages.append({"role": speaker, "content": dialogo[i]["text"]})

        # For fine-tuning, usually each training example is a complete conversation or a turn
        # Here, we will append each patient-doctor turn as a separate conversational example.
        # The original code iterated through pairs, so we'll maintain that logic for creating training examples.
        for i in range(len(conversation_messages) - 1):
            if conversation_messages[i]["role"] == "user" and conversation_messages[i+1]["role"] == "assistant":
                # Each training example is a user turn followed by an assistant turn
                dataset1.append({"messages": [conversation_messages[i], conversation_messages[i+1]]})
    return dataset1

# Generar dataset
dataset1 = dialogos_a_dataset(num_dialogos=20)

# Guardar resultados
filename = "dataset_dialogos_medicos_es.json"
with open(filename, "w", encoding="utf-8") as f:
    json.dump(dataset1, f, ensure_ascii=False, indent=2)

print(f"\nDataset guardado como '{filename}' (total: {len(dataset1)} ejemplos).")
print("Descarga el archivo desde el panel izquierdo de Colab (sección 'Files').")


Dataset guardado como 'dataset_dialogos_medicos_es.json' (total: 78 ejemplos).
Descarga el archivo desde el panel izquierdo de Colab (sección 'Files').


In [ ]:
print(dataset1)

[{'messages': [{'role': 'user', 'content': 'Doctor, tengo pérdida del olfato.'}, {'role': 'assistant', 'content': '¿Tienes alergias a algún medicamento?'}]}, {'messages': [{'role': 'user', 'content': 'Sí, tomé omeprazol.'}, {'role': 'assistant', 'content': '¿Desde cuándo tienes pérdida del olfato?'}]}, {'messages': [{'role': 'user', 'content': 'No, no tengo alergias.'}, {'role': 'assistant', 'content': '¿El pérdida del olfato es constante o viene y va?'}]}, {'messages': [{'role': 'user', 'content': 'Sí, viajé a la montaña.'}, {'role': 'assistant', 'content': 'Necesito revisar tu temperatura.'}]}, {'messages': [{'role': 'user', 'content': 'Me duele mucho náuseas.'}, {'role': 'assistant', 'content': '¿Has tenido fiebre?'}]}, {'messages': [{'role': 'user', 'content': 'No, no tengo condiciones médicas.'}, {'role': 'assistant', 'content': '¿Has viajado recientemente?'}]}, {'messages': [{'role': 'user', 'content': 'Sí, estuve con alguien enfermo.'}, {'role': 'assistant', 'content': '¿Has tom

In [ ]:
from huggingface_hub import login
from google.colab import userdata
import os

# Attempt to retrieve the HF_TOKEN from Colab secrets
hf_token = userdata.get('HF_TOKEN') # Changed to HF_TOKEN

if hf_token:
    # Log in to Hugging Face Hub
    login(token=hf_token)
    print("Successfully logged into Hugging Face Hub.")
else:
    print("Warning: HF_TOKEN not found in Colab secrets. Please add it as instructed above for more stable model loading.")

Successfully logged into Hugging Face Hub.


In [ ]:
# # Instalar bitsandbytes para cuantización de 8 bits
# !pip install bitsandbytes==0.41.3 --quiet
# !pip install transformers huggingface_hub --quiet accelerate==0.21.0 --quiet

# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, BitsAndBytesConfig
# from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
# from datasets import Dataset
# import json

# # Clear GPU memory from previous models if any
# if 'model' in globals() and hasattr(model, 'to'):
#     try:
#         model.to('cpu') # Move model to CPU before deletion to ensure GPU memory is freed
#         del model
#     except Exception:
#         pass # Model might already be on CPU or already deleted
# if 'tokenizer' in globals():
#     del tokenizer
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()

# # Configuración para cargar el modelo en 8 bits
# bnb_config = BitsAndBytesConfig(
#     load_in_8bit=True,
#     llm_int8_threshold=6.0,
#     llm_int8_enable_fp32_cpu_offload=True # Added to allow CPU offload for 32-bit modules
# )

# # Define the model identifier as a string
# model_id = "BioMistral/BioMistral-7B"

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

# Define the local path to your downloaded model on Google Drive
# IMPORTANT: Make sure this path matches where you uploaded the BioMistral-7B folder
#LOCAL_MODEL_PATH = '/content/drive/MyDrive/BioMistral-7B'
#.0.LOCAL_MODEL_PATH = '/content/drive/MyDrive/UNIR/Seminario_de_Innovacion/BioMistral'
0

0

In [ ]:
#!pip uninstall -y torch transformers peft bitsandbytes accelerate datasets

In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 60.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="BioMistral/BioMistral-7B")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/14.5G [00:00<?, ?B/s]

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 72, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 48, in spawn_con

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': " I'm Mistral, a language model trained by the Mistral AI team."}]}]

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("BioMistral/BioMistral-7B")
model = AutoModelForCausalLM.from_pretrained("BioMistral/BioMistral-7B")

# To perform inference with a single prompt, create a list of messages for that prompt.
# For example, to use the first user message from dataset1:
if dataset1 and dataset1[0] and 'messages' in dataset1[0] and dataset1[0]['messages']:
    # Take the user's part of the first conversation turn from dataset1
    # Note: For actual inference, you would typically provide a new prompt, not necessarily from the training data.
    # This example demonstrates how to correctly format a prompt from dataset1 for inference.
    inference_messages = [dataset1[0]['messages'][0]] # Take only the user's message from the first training example
else:
    # Fallback if dataset1 is empty or not in expected format
    inference_messages = [{
        "role": "user",
        "content": "Doctor, tengo dolor de cabeza."
    }]

inputs = tokenizer.apply_chat_template(
	inference_messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 72, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "/usr/local/lib/python3.12/dist-pac

¿Cuánto tiempo tienes esta pérdida del olfato? ¿Tienes fiebre? ¿Tienes tos o dificultad para


In [ ]:
!pip uninstall numpy -y
!pip uninstall torch torchvision torchaudio -y
!pip install torch==2.2.2 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip uninstall torch_xla -y # Explicitly uninstall torch_xla to prevent conflicts
!pip install transformers==4.40.0 peft==0.10.0 accelerate==0.29.3 datasets==2.18.0 xformers # bitsandbytes will be installed separately

Found existing installation: numpy 2.4.3
Uninstalling numpy-2.4.3:
  Successfully uninstalled numpy-2.4.3
Found existing installation: torch 2.11.0
Uninstalling torch-2.11.0:
  Successfully uninstalled torch-2.11.0
Found existing installation: torchvision 0.17.2+cu121
Uninstalling torchvision-0.17.2+cu121:
  Successfully uninstalled torchvision-0.17.2+cu121
Found existing installation: torchaudio 2.2.2+cu121
Uninstalling torchaudio-2.2.2+cu121:
  Successfully uninstalled torchaudio-2.2.2+cu121
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download.pytorch.org/whl/cu121/torch-2.2.2%2Bcu121-cp312-cp312-linux_x86_64.whl (757.2 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-linux_x86_64.whl (7.3 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (3.4 MB)
  Using cached numpy-2.4.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl

Found existing installation: bitsandbytes 0.43.0
Uninstalling bitsandbytes-0.43.0:
  Successfully uninstalled bitsandbytes-0.43.0
  Using cached torch-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.6/92.6 MB 9.8 MB/s eta 0:00:00
Using cached torch-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl (530.7 MB)
  Attempting uninstall: torch
    Found existing installation: torch 2.2.2+cu121
    Uninstalling torch-2.2.2+cu121:
      Successfully uninstalled torch-2.2.2+cu121
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.2.2+cu121 requires torch==2.2.2, but you have torch 2.11.0 which is incompatible.
torchvision 0.17.2+cu121 requires torch==2.2.2, but you have torch 2.11.0 which is incompatible.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 wh

In [ ]:
# Verificar la disponibilidad de la GPU
import torch
print("GPU disponible:", torch.cuda.is_available())
print("Nombre de la GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')

# Ruta local del modelo
LOCAL_MODEL_PATH = "/content/drive/MyDrive/UNIR/Seminario_de_Innovacion/BioMistral"

# Removed redundant !pip install commands, as F05gImQVqEsn should handle all installations.
# !pip install accelerate==0.27.0 transformers==4.36.2

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import json

# Configuración para 8-bit
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=True
)

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH, local_files_only=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'}) # Adding pad token if missing

model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_PATH,
    quantization_config=bnb_config,
    torch_dtype=torch.float16, # Keeping float16 for memory efficiency
    device_map="auto",
    trust_remote_code=True,
    local_files_only=True,
)

# Configurar LoRA
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], # Target specific modules for LoRA
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

# Cargar el dataset
with open('dataset_dialogos_medicos_es.json', 'r', encoding='utf-8') as f:
    raw_dataset = json.load(f)

# Formatear el dataset para el chat template
def format_conversation_for_training(example):
    # example will now be something like {'messages': [{'role': 'user', ...}, {'role': 'assistant', ...}]}
    # Apply the chat template to this list of messages to get a single formatted string
    # add_generation_prompt=False because we are providing the full conversation for training.
    formatted_text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": formatted_text}

formatted_dataset = [format_conversation_for_training(item) for item in raw_dataset]

# Convertir a Dataset de Hugging Face
dataset_dict = {"text": [item["text"] for item in formatted_dataset]}
dataset = Dataset.from_dict(dataset_dict)

# Tokenizar el dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Configuración del entrenamiento
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit", # Reverted optimizer back to paged_adamw_8bit
    save_steps=100,
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True, # Keep fp16 for training to save memory
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
    report_to="none",
)

# Configurar el Trainer
trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    tokenizer=tokenizer,
)

# Entrenar el modelo
trainer.train()

# Guardar el modelo
trainer.save_model("./fine_tuned_biomistral")

GPU disponible: True
Nombre de la GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
False

===================================BUG REPORT===================================
The following directories listed in your path were found to be non-existent: {PosixPath('/usr/local/lib/python3.12/dist-packages/cv2/../../lib64')}
The following directories listed in your path were found to be non-existent: {PosixPath('/sys/fs/cgroup/memory.events /var/colab/cgroup/jupyter-children/memory.events')}
The following directories listed in your path were found to be non-existent: {PosixPath('//mp.kaggle.net'), PosixPath('https')}
The following directories listed in your path were found to be non-existent: {PosixPath('//172.28.0.1'), PosixPath('8013'), PosixPath('http')}
The following directories listed in your path were found to be non-existent: {PosixPath('--logtostderr --listen_host=172.28.0.12 --target_host=172

/usr/local/lib/python3.12/dist-packages/bitsandbytes/cuda_setup/main.py:166: UserWarning: Welcome to bitsandbytes. For bug reports, please run

python -m bitsandbytes


  warn(msg)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/cuda_setup/main.py:166: UserWarning: /usr/local/lib/python3.12/dist-packages/cv2/../../lib64:/usr/lib64-nvidia did not contain ['libcudart.so', 'libcudart.so.11.0', 'libcudart.so.12.0'] as expected! Searching further paths...
  warn(msg)


RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):

        CUDA Setup failed despite GPU being available. Please run the following command to get more information:

        python -m bitsandbytes

        Inspect the output of the command and see if you can locate CUDA libraries. You might need to add them
        to your LD_LIBRARY_PATH. If you suspect a bug, please take the information from python -m bitsandbytes
        and open an issue at: https://github.com/TimDettmers/bitsandbytes/issues

In [ ]:
 python -m bitsandbytes

SyntaxError: invalid syntax (418472553.py, line 1)

In [ ]:
# Instalar bitsandbytes desde una fuente confiable
!pip install https://github.com/jllllll/bitsandbytes/releases/download/0.41.1/bitsandbytes-0.41.1-py3-none-any.whl

# Instalar accelerate
!pip install accelerate==0.27.0

# Verificar la instalación
import bitsandbytes as bnb
print("bitsandbytes versión:", bnb.__version__)

# Importar librerías
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Configuración para 8-bit
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

# Ruta local del modelo
LOCAL_MODEL_PATH = "/content/drive/MyDrive/UNIR/Seminario_de_Innovacion/BioMistral"

# Cargar el tokenizer
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH, local_files_only=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# Cargar el modelo
model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_PATH,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)


In [ ]:
# Probar el modelo
def generate_response(prompt):
    # Determine the device the model is currently on
    device = model.device if hasattr(model, 'device') else ("cuda" if torch.cuda.is_available() else "cpu")
    if device == "cpu" and torch.cuda.is_available():
        print("Warning: Model is on CPU but CUDA is available. Please ensure GPU runtime is selected and model is loaded correctly.")
    elif device == "cpu" and not torch.cuda.is_available():
        print("Running on CPU. Consider switching to a GPU runtime for better performance.")

    inputs = tokenizer(f"<s>[INST] {prompt} [/INST]", return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_new_tokens=256)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.replace("</s>", "").replace("<s>", "").replace("[INST]", "").replace("[/INST]", "")
    return response


# Evaluar el rendimiento del modelo con nuevos prompts
print("\n--- Evaluación del Modelo ---")

# Ejemplo 1
prompt1 = "Doctor, he tenido fiebre y dolor de garganta durante dos días."
response1 = generate_response(prompt1)
print(f"Paciente: {prompt1}\nMédico: {response1}\n")

# Ejemplo 2
prompt2 = "Doctor, me duele el estómago y tengo náuseas después de comer."
response2 = generate_response(prompt2)
print(f"Paciente: {prompt2}\nMédico: {response2}\n")

# Ejemplo 3
prompt3 = "He estado tosiendo mucho y me siento muy fatigado."
response3 = generate_response(prompt3)
print(f"Paciente: {prompt3}\nMédico: {response3}\n")

In [ ]:
# from flask import Flask, request, jsonify
# app = Flask(__name__)

# @app.route('/chat', methods=['POST'])
# def chat():
#     data = request.json
#     prompt = data.get('message', '')
#     inputs = tokenizer(f"<s>[INST] {prompt} [/INST]", return_tensors="pt").to("cuda")
#     outputs = model.generate(**inputs, max_new_tokens=256)
#     response = tokenizer.decode(outputs[0], skip_special_tokens=True).replace("</s>", "").replace("<s>", "").replace("[INST]", "").replace("[/INST]", "")
#     return jsonify({"response": response})

# if __name__ == '__main__':
#     app.run(host='0.0.0.0', port=5000)